<div align="center">

# Hands-on Workshop on Recent Advances in Cellular Automata-Based Wildfire Simulation with Cell2Fire


**Dr. Jaime Carrasco-Barra**  
Metropolitan Technological University (UTEM),  and \
Complex Engineering Systems Institute (ISCI), Chile  

[![GitHub](https://img.shields.io/badge/GitHub-C2F--W-black?logo=github)](https://github.com/fire2a/C2F-W)

**AUTOMATA / ACRI 2026 — Hands-on Workshop**

---
---

<div align="center">

# Activity 1 — Simulating a wildfire with Cell2Fire (30 min)

</div>


In this activity you will:

0. SETUP: Cell2Fire and data instances
1. Run a single wildfire simulation on the `vilo100` landscape located in Catalonia/Spain (Scott & Burgan fuels).
2. Explore how the **moisture scenario `DkLm`** changes the fire (D = dead, L = live; 1 = dry … 4 = wet).
3. Use an **interactive map**: click a point to set the ignition and simulate from there (a proxy for the QGIS plugin).

> Everything runs in Colab — no local install needed.

This notebook builds on the Cell2Fire simulation framework proposed by [Carrasco-Barra et al. (2026)](https://doi.org/10.1016/j.envsoft.2026.107009) and the multicriteria firebreak planning framework presented by [Carrasco-Barra et al. (2025)](https://doi.org/10.1016/j.indic.2025.100956).

---

## 0. SETUP: Cell2Fre and data instances

### 0.1 Cell2Fire installation

In [ ]:
# === CONFIG: where to get the instances and the engine =====================
C2FW_REPO     = "https://github.com/fire2a/C2F-W"
C2FW_BRANCH   = "main"
INSTANCES_PATH = "data/handson"      # instances live inside the repo

# Engine source. The v2.0 (continuous-moisture) engine is required for --moisture-scenario.
#   "drive"  -> download a prebuilt source zip from Google Drive (works on Colab today)
#   "github" -> build from a branch of C2F-W that already contains the v2.0 engine (preferred)
ENGINE_SOURCE   = "drive"
ENGINE_BRANCH   = "feat/sb-continuous-moisture"        # used only if ENGINE_SOURCE == "github"
SRC_DRIVE_ID    = "1bpt0EUadMi-d97I5_Ix-IQoFm_MswkAi"  # c2fw_v2_src.zip (used if "drive")
# ===========================================================================
import os, sys, subprocess, shutil, zipfile
from pathlib import Path
MAT = Path.cwd()/"c2f_be"; MAT.mkdir(exist_ok=True)

if shutil.which("g++") is None or shutil.which("make") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y g++ make libtiff-dev libboost-random-dev",
                   shell=True, check=False)

# 1) instances: sparse-checkout just the Instances_Belgium folder from the repo
repo = MAT/"C2F-W"
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","--filter=blob:none","--sparse",
                    "-b",C2FW_BRANCH,C2FW_REPO,str(repo)],check=True)
    subprocess.run(["git","sparse-checkout","set",INSTANCES_PATH],cwd=str(repo),check=True)
BE = repo/INSTANCES_PATH
CASES = sorted([p.name for p in BE.iterdir() if p.is_dir()])
print("instances from", C2FW_REPO, "->", CASES)

# 2) engine
if ENGINE_SOURCE == "github":
    eng = MAT/"engine"
    if not eng.exists():
        subprocess.run(["git","clone","--depth","1","--filter=blob:none","--sparse",
                        "-b",ENGINE_BRANCH,C2FW_REPO,str(eng)],check=True)
        subprocess.run(["git","sparse-checkout","set","Cell2Fire"],cwd=str(eng),check=True)
    SRC_DIR = eng/"Cell2Fire"
else:
    subprocess.run([sys.executable,"-m","pip","install","-q","gdown"]); import gdown
    gdown.download(id=SRC_DRIVE_ID, output=str(MAT/"src.zip"), quiet=False)
    with zipfile.ZipFile(MAT/"src.zip") as z: z.extractall(MAT)
    SRC_DIR = MAT/"Cell2Fire"
subprocess.run(["make","clean"],cwd=str(SRC_DIR),check=False)
subprocess.run(["make"],cwd=str(SRC_DIR),check=True)
EXEC = str(SRC_DIR/"Cell2Fire"); os.environ["C2F_EXEC"]=EXEC
print("engine:", EXEC, "| exists:", Path(EXEC).exists())

In [ ]:
# --- helper: fetch a reference figure from Google Drive and display it (cached) -----------
def show_drive_image(file_id, path, width=800):
    import os, subprocess
    from IPython.display import Image, display     # local import: immune to later `from PIL import Image`
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if not os.path.exists(path):
        subprocess.run(["wget", "-q", "-O", path,
                        f"https://drive.google.com/uc?export=download&id={file_id}"], check=False)
    display(Image(path, width=width))


### 0.2 Data instances

`vilo100` lanscape is a 100×100 grid at 20 m resolution (2 km × 2 km). The S&B engine reads, per cell:
the **fuel model**, **elevation**, **slope** (`ps`, %), **up‑slope azimuth** (`saz`), and
**canopy cover** (`ccf`). These four terrain/fuel layers drive everything that follows.

In [ ]:
!ls c2f_be/C2F-W/data/handson/vilo100/


In [ ]:
# landscape variables
# https://drive.google.com/file/d/1fB9AA5t2udDmqvTC5BAtWE0cG9CeTYgi/view?usp=drive_link
show_drive_image("1fB9AA5t2udDmqvTC5BAtWE0cG9CeTYgi", "/content/figures/data_for_fire_modeling.png", width=800)


In [ ]:
# Scott & Burgan fuel models (or class)
# https://drive.google.com/file/d/1f3Wie9JOAut6Ht3Jmfc3dR_613zdLLei/view?usp=drive_link
show_drive_image("1f3Wie9JOAut6Ht3Jmfc3dR_613zdLLei", "/content/figures/scott_burgan_fuel_models.png", width=800)

In [ ]:
!cat c2f_be/C2F-W/data/handson/vilo100/Weather.csv | column -s, -t

In [ ]:
# --- helpers ---------------------------------------------------------------
import csv
import numpy as np
import matplotlib.pyplot as plt


def read_asc(path):
    # Read an ESRI ASCII grid -> (masked 2D array, header dict).
    header = {}
    with open(path) as f:
        for _ in range(6):
            k, v = f.readline().split()
            header[k.lower()] = float(v)
    data = np.loadtxt(path, skiprows=6)
    return np.ma.masked_equal(data, header.get("nodata_value", -9999)), header

## --- a fuel colormap from the lookup table (nice fuel maps) -----------------
def fuel_rgb_image(fuels):
    lut = {}
    lp = INST / "spain_lookup_table.csv"
    if lp.exists():
        with open(lp) as f:
            for row in csv.DictReader(f):
                try:
                    lut[int(float(row["grid_value"]))] = (
                        int(row[" r"]) / 255, int(row[" g"]) / 255, int(row[" b"]) / 255)
                except Exception:
                    pass
    img = np.full(fuels.shape + (3,), 0.85)  # light grey default
    for v in np.unique(fuels.filled(-1)):
        if int(v) in lut:
            img[fuels.filled(-1) == v] = lut[int(v)]
    return img


### 0.3 Visualizaion of the landscape data (vilo100)

In [ ]:
INST = Path("c2f_be/C2F-W/data/handson/vilo100")
fuels, h = read_asc(INST / "fuels.asc")
elev, _  = read_asc(INST / "elevation.asc")
slope, _ = read_asc(INST / "slope.asc")
saz, _   = read_asc(INST / "saz.asc")
ccf, _   = read_asc(INST / "ccf.asc")

fig, ax = plt.subplots(2, 3, figsize=(13, 8))
ax[0,0].imshow(fuel_rgb_image(fuels));           ax[0,0].set_title("Fuel model (S&B)")
im=ax[0,1].imshow(elev, cmap="terrain");          ax[0,1].set_title("Elevation (m)"); plt.colorbar(im,ax=ax[0,1],shrink=.8)
im=ax[0,2].imshow(slope, cmap="magma");           ax[0,2].set_title("Slope (%)");     plt.colorbar(im,ax=ax[0,2],shrink=.8)
im=ax[1,0].imshow(saz, cmap="twilight");          ax[1,0].set_title("Up-slope azimuth (deg)"); plt.colorbar(im,ax=ax[1,0],shrink=.8)
im=ax[1,1].imshow(ccf, cmap="Greens");            ax[1,1].set_title("Canopy cover (%)"); plt.colorbar(im,ax=ax[1,1],shrink=.8)
ax[1,2].axis("off")
for a in ax.ravel()[:5]: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print("grid:", int(h["ncols"]), "x", int(h["nrows"]), "| cellsize", h["cellsize"], "m")

---

## 1. Run a single wildfire simulation on the `vilo100` landscape located in Catalonia/Spain (Scott & Burgan fuels)

*CELL2FIRE (C2F-W) – RULES OF SPATIAL DYNAMICS:*

In [ ]:
# https://drive.google.com/file/d/1K5HEof7FFlEE0zv3l_RWmerAV0Wilu-U/view?usp=drive_link
show_drive_image("1K5HEof7FFlEE0zv3l_RWmerAV0Wilu-U", "/content/figures/cellular_huygens_principle.png", width=800)

# The cellular spread rules are as follows: when a cell ignites,
# an elliptical fire perimeter expands toward its neighboring cells.
# This ellipse is built by placing the center of the burning cell at one of its foci.
# When the ellipse reaches the center of another cell, that cell ignites and starts generating its own fire ellipse.
# The process then continues from cell to cell.


### 1.1 Visualization of the underlying Cellular Huygens Principle in Cell2Fire


In [ ]:
# Helper functions 1

def write_weather(n, ws, wd, T=None, RH=None, datetime="15/07/13 13:00"):
    # Write a constant Weather.csv with n periods. 4 cols (WS,WD) or 6 cols (+T,RH).
    p = INST / "Weather.csv"
    with open(p, "w") as f:
        if T is None:
            f.write("Instance,datetime,WS,WD\n")
            for _ in range(n): f.write(f"vilo,{datetime},{ws},{wd}\n")
        else:
            f.write("Instance,datetime,WS,WD,T,RH\n")
            for _ in range(n): f.write(f"vilo,{datetime},{ws},{wd},{T},{RH}\n")
    return p

def set_ignition(cell):
    (INST / "Ignitions.csv").write_text(f"Year,Ncell\n1,{cell}\n")


In [ ]:
# Helper functions 2

import math
from matplotlib.patches import Ellipse, FancyArrow
from PIL import Image
from IPython.display import Image as IImage

def animate_messages(out, gif_path, *, seeds, frames_n=56, dpi=130, colors=200, minutes=None):
    NCOLS, NROWS = int(h["ncols"]), int(h["nrows"])
    arcs = []
    with open(out / "Messages" / "MessagesFile1.csv") as f:
        for r in csv.reader(f):
            if r and len(r) >= 4:
                arcs.append((int(r[0]), int(r[1]), float(r[2]), float(r[3])))
    ign = {}
    for i, j, tj, _ in arcs:
        ign[j] = min(ign.get(j, 1e18), tj)
    for s in seeds:
        ign[s] = 0.0
    rv = [a[3] for a in arcs]; r0, r1 = min(rv), max(rv)
    arcs_out = [a for a in arcs if a[1] not in seeds]
    bg = fuel_rgb_image(fuels)
    WD = float(list(csv.DictReader(open(INST / "Weather.csv")))[0]["WD"])
    Tig = np.full((NROWS, NCOLS), np.nan)              # ignition-time grid (vectorised burned layer)
    for c, ti in ign.items():
        Tig[(c - 1) // NCOLS, (c - 1) % NCOLS] = ti
    def xy(c):
        r = (c - 1) // NCOLS; cc = (c - 1) % NCOLS; return cc + 0.5, NROWS - r - 0.5
    def ecc(ros):
        return 0.30 + 0.55 * (ros - r0) / (r1 - r0 + 1e-9)
    DARK = np.array([0x2a, 0x1c, 0x16]) / 255
    FLASH = np.array([1.0, 0.48, 0.15]); EDGE = (0xD8 / 255, 0x5A / 255, 0x30 / 255)
    tmax = np.nanmax(Tig); T = minutes if minutes else tmax * 1.04
    times = list(np.linspace(0, T, frames_n)) + [T] * 8
    fig, ax = plt.subplots(figsize=(7.2, 7.4), dpi=dpi)
    def burned_rgba(t):
        b = (Tig <= t); age = t - Tig; img = np.zeros((NROWS, NCOLS, 4))
        img[b, :3] = DARK; img[b, 3] = 0.58
        fl = b & (age < 18); img[fl, :3] = FLASH; img[fl, 3] = 0.82 * (1 - age[fl] / 18)
        return img
    def draw(t):
        ax.cla(); ax.set_xlim(0, NCOLS); ax.set_ylim(0, NROWS); ax.set_aspect("equal"); ax.axis("off")
        ax.imshow(bg, extent=[0, NCOLS, 0, NROWS], origin="upper", interpolation="nearest", zorder=0)
        ax.imshow(burned_rgba(t), extent=[0, NCOLS, 0, NROWS], origin="upper", interpolation="nearest", zorder=2)
        for i, j, tj, ros in arcs_out:                # only the active wavelets (leading edge) are drawn
            ti = ign[i]
            if ti <= t < tj:
                xi, yi = xy(i); xj, yj = xy(j); dx, dy = xj - xi, yj - yi
                dist = math.hypot(dx, dy); ang = math.atan2(dy, dx)
                Rh = dist * (t - ti) / (tj - ti + 1e-9)
                if Rh < 1e-3:
                    continue
                e = ecc(ros); a = Rh / (1 + e); b = a * math.sqrt(1 - e * e); c = a * e
                ax.add_patch(Ellipse((xi + c * math.cos(ang), yi + c * math.sin(ang)), 2 * a, 2 * b,
                             angle=math.degrees(ang), facecolor=(*EDGE, 0.05), edgecolor=(*EDGE, 0.97),
                             lw=1.8, zorder=4, antialiased=True))
        for s in seeds:
            xs, ys = xy(s); ax.plot(xs, ys, marker="*", ms=12, color="#FFD23F", mec="#7a3a16", mew=0.8, zorder=7)
        A = math.radians(WD + 180.0); ox, oy = 8.0, NROWS - 3.0
        ax.add_patch(FancyArrow(ox, oy, math.sin(A) * 3, math.cos(A) * 3, width=0.15, head_width=0.9,
                     head_length=0.9, length_includes_head=True, color="#1D7F9E", zorder=8))
        ax.text(ox, oy + 1.8, f"wind from {WD:.0f}\u00b0", fontsize=8, color="#1D7F9E", ha="center")
        ax.set_title(f"Huygens elliptical wavelets  \u00b7  t = {t:5.1f} min  \u00b7  "
                     f"{int(np.sum(Tig <= t))} cells burned", fontsize=11, fontweight="bold", pad=8)
        fig.subplots_adjust(left=0.02, right=0.98, top=0.95, bottom=0.02)
    frames = []
    for t in times:
        draw(t); fig.canvas.draw()
        frames.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba()))
                      .convert("RGB").convert("P", palette=Image.ADAPTIVE, colors=colors))
    plt.close(fig)
    frames[0].save(gif_path, save_all=True, append_images=frames[1:],
                   duration=120, loop=0, optimize=False, disposal=2)
    return gif_path

In [ ]:
# Run 2D
WORK = Path("/tmp/c2f_handson")
out  = WORK / "runs" / "anim"
if out.exists(): shutil.rmtree(out)
out.mkdir(parents=True)

set_ignition(5051)
write_weather(14, ws=18, wd=0)                       # W wind -> fire heads east

cmd = [str(EXEC), "--input-instance-folder", str(INST)+"/", "--output-folder", str(out)+"/",
       "--sim", "S", "--ignitions", "--seed", "1", "--nsims", "1", "--final-grid",
       "--output-messages", "--out-fl","--moisture-scenario", "D2L3",
       "--Fire-Period-Length", "1.0", "--Weather-Period-Length", "60", "--max-fire-periods", str(60*4)]
r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-1500:]); print(r.stderr[-1500:])
    raise RuntimeError(f"Cell2Fire failed (code={r.returncode})")
out_anim = out

gif = animate_messages(out_anim, str(WORK / "huygens.gif"), seeds={5051})
print("gif:", gif)
IImage(open(gif, "rb").read())


### 1.3 Fire propagation in 3D (draped on the terrain)  + flame length

In [ ]:
try:
    import plotly.graph_objects as go
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"]); import plotly.graph_objects as go

# arrival-time grid + per-cell flame length (run the sim with  --output-messages --out-fl)
NCOLS, NROWS = int(h["ncols"]), int(h["nrows"])
_arcs = []
with open(out_anim / "Messages" / "MessagesFile1.csv") as f:
    for r in csv.reader(f):
        if r and len(r) >= 4:
            _arcs.append((int(r[0]), int(r[1]), float(r[2])))
_ign = {5051: 0.0}
for i, j, tj in _arcs:
    _ign[j] = min(_ign.get(j, 1e18), tj)
Tig3 = np.full((NROWS, NCOLS), np.nan)
for c, ti in _ign.items():
    Tig3[(c - 1) // NCOLS, (c - 1) % NCOLS] = ti

FL3 = np.loadtxt(out_anim / "SurfaceFlameLength" / "SurfaceFlameLength1.asc", skiprows=6)
FL3 = np.where(FL3 <= -9999, 0.0, FL3)                  # flame length (m) per cell

s = 2                                                  # downsample -> light, smooth 3D
Z = elev.filled(np.nan)[::s, ::s]; A = Tig3[::s, ::s]; FLa = FL3[::s, ::s]
Xg = np.arange(0, NCOLS, s) * h["cellsize"]; Yg = np.arange(0, NROWS, s) * h["cellsize"]
tmax = np.nanmax(Tig3); steps = np.linspace(tmax * 0.08, tmax, 14)

# colour the fire by FLAME LENGTH: low = yellow, high = deep red
FL_MAX = np.nanpercentile(FLa[np.isfinite(A) & (FLa > 0)], 95)   # clip outliers so high FL reads red
FL_SCALE = [[0.0, "#ffec99"], [0.25, "#ffc14d"], [0.5, "#ff8c1a"], [0.75, "#f4511e"], [1.0, "#b71c1c"]]

def _fz(t): return np.where(~np.isnan(A) & (A <= t), Z + 4, np.nan)   # fire drape, just above ground
def _fc(t): return np.where(~np.isnan(A) & (A <= t), FLa, np.nan)     # colour = flame length

terrain = go.Surface(x=Xg, y=Yg, z=Z, surfacecolor=Z, colorscale="earth", showscale=False,
                     opacity=1.0, lighting=dict(ambient=0.7, diffuse=0.6), name="terrain")
fire0 = go.Surface(x=Xg, y=Yg, z=_fz(steps[0]), surfacecolor=_fc(steps[0]), colorscale=FL_SCALE,
                   cmin=0, cmax=FL_MAX, colorbar=dict(title="flame<br>length (m)", len=0.6))
frames = [go.Frame(data=[go.Surface(z=_fz(t), surfacecolor=_fc(t))], traces=[1], name=f"{t:.0f}")
          for t in steps]
fig = go.Figure(data=[terrain, fire0], frames=frames)
fig.update_layout(height=620, margin=dict(l=0, r=0, t=40, b=0),
    title="Fire propagation over the terrain (3D) — colour = flame length",
    scene=dict(aspectmode="manual", aspectratio=dict(x=1, y=1, z=0.45),
               xaxis_title="E (m)", yaxis_title="N (m)", zaxis_title="elev (m)",
               camera=dict(eye=dict(x=1.5, y=-1.5, z=0.9))),
    updatemenus=[dict(type="buttons", showactive=False, x=0.02, y=0.9, buttons=[
        dict(label="Play", method="animate",
             args=[None, {"frame": {"duration": 260, "redraw": True}, "fromcurrent": True}]),
        dict(label="Pause", method="animate",
             args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])])],
    sliders=[dict(active=0, y=0, x=0.12, len=0.85, currentvalue=dict(prefix="t = ", suffix=" min"),
        steps=[dict(method="animate", label=f"{t:.0f}",
                    args=[[f"{t:.0f}"], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}])
               for t in steps])])
fig   # interactive 3D: drag to rotate, scroll to zoom, Play / slider to animate

### 1.4 Use an interactive map: click a point to set the ignition and simulate from there (a proxy for the QGIS plugin)

Click anywhere on the map to set the ignition point. Cell2Fire runs from there:
the **final scar** is drawn on the map, and the **elliptical propagation front** (Huygens
wavelets, one ellipse per advancing cell) is animated just below.
Pick the moisture scenario from the dropdown. (This mimics the QGIS plugin workflow.)

#### Mini Plugin

In [ ]:
# @title
# mini-plugin for visualization
import numpy as np, csv, io, base64, subprocess, shutil, tempfile
import pyproj, IPython
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import output as colab_output

CRS_VILO="EPSG:25831"
vilo = Path("c2f_be/C2F-W/data/handson/vilo100")
_, _h=read_asc(vilo/"fuels.asc"); _nc,_nr,_cs=int(_h["ncols"]),int(_h["nrows"]),_h["cellsize"]
_xll,_yll=_h["xllcorner"],_h["yllcorner"]; _fuel=np.loadtxt(vilo/"fuels.asc",skiprows=6)
_to_utm=pyproj.Transformer.from_crs("EPSG:4326",CRS_VILO,always_xy=True)
_to_wgs=pyproj.Transformer.from_crs(CRS_VILO,"EPSG:4326",always_xy=True)
def _dom_bounds():
    cor=[(_xll,_yll),(_xll+_nc*_cs,_yll),(_xll,_yll+_nr*_cs),(_xll+_nc*_cs,_yll+_nr*_cs)]
    ll=[_to_wgs.transform(x,y) for x,y in cor]; la=[p[1] for p in ll]; lo=[p[0] for p in ll]
    return min(la),max(la),min(lo),max(lo)
S,N,W,E=_dom_bounds()

def _fuel_overlay_uri():
    """Colour the landscape (fuel model) as a translucent overlay over the whole domain."""
    codes=np.unique(_fuel[_fuel>0]); cmap=plt.get_cmap("tab20", max(len(codes),1))
    rgba=np.zeros((_nr,_nc,4))
    for k,cval in enumerate(codes):
        m=_fuel==cval; col=cmap(k); rgba[m]=[col[0],col[1],col[2],0.45]
    bio=io.BytesIO(); Image.fromarray((rgba*255).astype("uint8")).save(bio,format="PNG")
    return "data:image/png;base64,"+base64.b64encode(bio.getvalue()).decode()
FUEL_URI=_fuel_overlay_uri()

# available Weather.csv files in the instance (if any)
_wdir=vilo/"Weathers"
# example weathers shipped with the instance (+ any generated in a Weathers/ folder)
_wf=sorted(vilo.glob("Weather_*.csv"))
if _wdir.exists(): _wf+=sorted(_wdir.glob("Weather*.csv"))
WEATHER_FILES=[p.name for p in _wf]

def _bounds_crop(r0,r1,c0,c1):
    x0=_xll+c0*_cs; x1=_xll+c1*_cs; y1=_yll+(_nr-r0)*_cs; y0=_yll+(_nr-r1)*_cs
    la,lo=[],[]
    for x in (x0,x1):
        for y in (y0,y1):
            o,a=_to_wgs.transform(x,y); lo.append(o); la.append(a)
    return [[min(la),min(lo)],[max(la),max(lo)]]

import re as _re
def run_isochrones(lat, lon, scenario="D2L2", mode="const", ws=15.0, wd=270.0,
                   nrows=5, weather_file=""):
    """Single Cell2Fire run -> arrival-time grid (minutes) for JS isochrone animation."""
    scenario=(scenario or "D2L2").strip().upper()
    if not _re.fullmatch(r"D[1-4]L[1-4]", scenario):
        return IPython.display.JSON({"ok":False,"msg":f"invalid scenario '{scenario}' - use DkLm (e.g. D2L2)"})
    x,y=_to_utm.transform(lon,lat)
    col=min(max(int((x-_xll)//_cs),0),_nc-1); row=min(max(int((_yll+_nr*_cs-y)//_cs),0),_nr-1)
    cell=row*_nc+col+1
    if _fuel[row,col]<=0:
        return IPython.display.JSON({"ok":False,"msg":f"cell {cell} is non-burnable - pick another point"})
    # weather
    if mode=="file" and weather_file:
        src=vilo/weather_file
        if not src.exists(): src=_wdir/weather_file
        if not src.exists(): return IPython.display.JSON({"ok":False,"msg":f"weather file not found: {weather_file}"})
        shutil.copy(src, vilo/"Weather.csv")
        n=max(sum(1 for _ in open(vilo/"Weather.csv"))-1,1)
    else:
        n=max(int(nrows),1)
        (vilo/"Weather.csv").write_text("Instance,datetime,WS,WD\n"+
            "\n".join(f"KitralSP,2024-07-15 {(14+i)%24:02d}:00,{ws},{int(wd)}" for i in range(n))+"\n")
    (vilo/"Ignitions.csv").write_text(f"Year,Ncell\n1,{cell}\n")
    out=Path(tempfile.gettempdir())/"iso"
    if out.exists(): shutil.rmtree(out)
    out.mkdir(parents=True)
    cmd=[EXEC,"--input-instance-folder",str(vilo)+"/","--output-folder",str(out)+"/","--sim","S","--ignitions",
         "--seed","1","--nsims","1","--final-grid","--output-messages","--moisture-scenario",scenario,
         "--Fire-Period-Length","1.0","--Weather-Period-Length","60","--max-fire-periods",str(n*60)]
    ok=False
    for _ in range(3):
        subprocess.run(cmd,capture_output=True,text=True)
        if (out/"Messages"/"MessagesFile1.csv").exists(): ok=True; break
    if not ok: return IPython.display.JSON({"ok":False,"msg":"no output - press Play again"})
    # arrival-time grid (minutes)
    arcs=[(int(r[0]),int(r[1]),float(r[2])) for r in csv.reader(open(out/"Messages"/"MessagesFile1.csv")) if len(r)>=4]
    ign={cell:0.0}
    for i,j,tj in arcs: ign[j]=min(ign.get(j,1e18),tj)
    T=np.full((_nr,_nc),np.nan)
    for c,t in ign.items(): T[(c-1)//_nc,(c-1)%_nc]=t
    rs,cx=np.where(np.isfinite(T)); m=6
    r0,r1=max(rs.min()-m,0),min(rs.max()+m+1,_nr); c0,c1=max(cx.min()-m,0),min(cx.max()+m+1,_nc)
    Tc=T[r0:r1,c0:c1]*1.0   # FPL=1 -> arrival index already equals minutes (index * FPL)
    tmax=float(np.nanmax(Tc)); ha=float(np.isfinite(Tc).sum()*_cs*_cs/1e4)
    # isochrone contour lines every 60 min (grid coords -> lat/lon)
    iso_lines=[]
    levels=list(range(60,int(tmax)+1,60))
    if levels:
        figc=plt.figure(); axc=figc.add_subplot(111)
        csr=axc.contour(np.where(np.isfinite(Tc),Tc,np.nan),levels=levels)
        for lev,segs in zip(csr.levels,csr.allsegs):
            longest=max((len(s) for s in segs),default=0)
            for s in segs:
                pts=[]
                for x,yv in s:
                    e=_xll+(c0+x+0.5)*_cs; nn=_yll+(_nr-(r0+yv)-0.5)*_cs
                    lo,la=_to_wgs.transform(e,nn); pts.append([la,lo])
                if len(pts)>=2:
                    iso_lines.append({"level":float(lev),"label":len(s)==longest,"pts":pts})
        plt.close(figc)
    flat=[(-1.0 if not np.isfinite(v) else round(float(v),2)) for v in Tc.ravel()]
    return IPython.display.JSON({"ok":True,"cell":int(cell),"ha":round(ha,1),
        "nr":int(Tc.shape[0]),"nc":int(Tc.shape[1]),"tmax":tmax,
        "times":flat,"bounds":_bounds_crop(r0,r1,c0,c1),"iso_lines":iso_lines})
colab_output.register_callback("fire.iso", run_isochrones)

_wopts='<option value="">(choose a file)</option>'+"".join(f'<option value="{w}">{w}</option>' for w in WEATHER_FILES)
_html="""
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<div style="font:13px sans-serif;margin-bottom:6px;line-height:2">
  <b>Ignition:</b> click the map &nbsp;|&nbsp;
  <b>Scenario</b> <input id="scn" type="text" value="D2L2" maxlength="4" style="width:56px" placeholder="D2L2">
  &nbsp; <b>hours</b> <input id="nrows" type="number" value="5" min="1" max="12" style="width:44px">
  <br>
  <b>Weather</b>
  <select id="wmode">
    <option value="const" selected>constant</option>
    <option value="file">from file</option>
  </select>
  &nbsp;WS <input id="ws" type="number" value="15" step="1" style="width:52px"> km/h
  &nbsp;WD <input id="wd" type="number" value="270" step="5" style="width:56px">&deg;
  &nbsp;file <select id="wfile">__WOPTS__</select>
  &nbsp; <button id="play" style="font-weight:bold;padding:3px 12px">&#9654; Play</button>
  &nbsp; <button id="reset" style="padding:3px 10px">Reset</button>
  &nbsp;<span id="info" style="color:#b23b3b;font-weight:bold"></span>
  <br>
  t = <input id="slider" type="range" min="0" max="100" value="100" style="width:280px;vertical-align:middle">
  <span id="tlab"></span>
  &nbsp;&nbsp;<label style="cursor:pointer"><input id="isoLines" type="checkbox"> isochrone lines (60 min)</label>
</div>
<div id="map" style="height:460px;border-radius:10px"></div>
<script>
var map=L.map('map').setView([__CLAT__,__CLON__],14);
L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
  {attribution:'Esri',maxZoom:19}).addTo(map);
var DB=[[__S__,__W__],[__N__,__E__]];
L.imageOverlay('__FUEL__',DB,{opacity:0.85}).addTo(map);          // the landscape (fuel map)
L.rectangle(DB,{color:'#19b3ff',weight:2,fill:false}).addTo(map);
map.fitBounds(DB);
var marker=null, iso=null, DATA=null, timer=null, latlng=null;
var isoGroup=L.layerGroup().addTo(map);
function clearIso(){ if(iso){ map.removeLayer(iso); iso=null; } }
function drawIsoLines(){
  isoGroup.clearLayers();
  if(!DATA || !DATA.iso_lines || !document.getElementById('isoLines').checked) return;
  DATA.iso_lines.forEach(function(s){
    var pl=L.polyline(s.pts,{color:'#111',weight:1.6,opacity:0.9}).addTo(isoGroup);
    if(s.label){ pl.bindTooltip(Math.round(s.level)+' min',{permanent:true,direction:'center',className:'isolab'}); }
    else { pl.bindTooltip(Math.round(s.level)+' min'); }
  });
}
map.on('click',function(e){ clearIso(); isoGroup.clearLayers(); DATA=null;
  if(marker) map.removeLayer(marker); marker=L.marker(e.latlng).addTo(map);
  latlng=e.latlng; document.getElementById('info').innerText='ignition set - press Play';
  document.getElementById('tlab').innerText=''; });
document.getElementById('isoLines').onchange=drawIsoLines;
document.getElementById('wfile').onchange=function(){ if(this.value) document.getElementById('wmode').value='file'; };

// YlOrRd-like gradient: early (0) = yellow, late (1) = dark red
function grad(u){
  var stops=[[255,255,178],[254,204,92],[253,141,60],[240,59,32],[128,0,38]];
  var x=Math.max(0,Math.min(1,u))*(stops.length-1); var i=Math.floor(x); var f=x-i;
  if(i>=stops.length-1) return stops[stops.length-1];
  var a=stops[i],b=stops[i+1];
  return [Math.round(a[0]+f*(b[0]-a[0])),Math.round(a[1]+f*(b[1]-a[1])),Math.round(a[2]+f*(b[2]-a[2]))];
}
function render(thr){
  if(!DATA) return;
  var nc=DATA.nc,nr=DATA.nr,cv=document.createElement('canvas'); cv.width=nc; cv.height=nr;
  var ctx=cv.getContext('2d'); var img=ctx.createImageData(nc,nr); var d=img.data;
  for(var k=0;k<nc*nr;k++){ var t=DATA.times[k]; var o=k*4;
    if(t>=0 && t<=thr){ var c=grad(DATA.tmax>0? t/DATA.tmax : 0); d[o]=c[0];d[o+1]=c[1];d[o+2]=c[2];d[o+3]=225; }
    else { d[o+3]=0; } }
  ctx.putImageData(img,0,0);
  var url=cv.toDataURL();
  if(iso){ iso.setUrl(url); } else { iso=L.imageOverlay(url,DATA.bounds,{opacity:0.9}).addTo(map); }
  document.getElementById('tlab').innerText=Math.round(thr)+' / '+Math.round(DATA.tmax)+' min';
}
function animate(){
  var i=0, steps=44; if(timer) clearInterval(timer);
  timer=setInterval(function(){ var thr=DATA.tmax*i/steps; render(thr);
    document.getElementById('slider').value=100*i/steps;
    if(i++>=steps){ clearInterval(timer); render(DATA.tmax); document.getElementById('slider').value=100; drawIsoLines(); }
  },90);
}
document.getElementById('slider').oninput=function(){ if(DATA) render(DATA.tmax*this.value/100); };
document.getElementById('play').onclick=async function(){
  if(!latlng){ document.getElementById('info').innerText='click the map first'; return; }
  if(timer) clearInterval(timer);
  document.getElementById('info').innerText='running Cell2Fire...';
  var args=[latlng.lat, latlng.lng,
            document.getElementById('scn').value, document.getElementById('wmode').value,
            parseFloat(document.getElementById('ws').value), parseFloat(document.getElementById('wd').value),
            parseInt(document.getElementById('nrows').value), document.getElementById('wfile').value];
  var res=await google.colab.kernel.invokeFunction('fire.iso',args,{});
  var d=res.data['application/json'];
  if(!d.ok){ document.getElementById('info').innerText=d.msg; return; }
  clearIso();                       // recreate overlay with the NEW crop bounds
  DATA=d; document.getElementById('info').innerText='cell '+d.cell+' - '+d.ha+' ha - tmax '+Math.round(d.tmax)+' min';
  animate();
};
document.getElementById('reset').onclick=function(){
  if(timer) clearInterval(timer);
  clearIso(); isoGroup.clearLayers(); DATA=null; latlng=null;
  if(marker){ map.removeLayer(marker); marker=null; }
  document.getElementById('info').innerText='reset - click the map';
  document.getElementById('tlab').innerText=''; document.getElementById('slider').value=100;
};
</script>
"""
_html=(_html.replace("__WOPTS__",_wopts).replace("__FUEL__",FUEL_URI)
            .replace("__CLAT__",str((S+N)/2)).replace("__CLON__",str((W+E)/2))
            .replace("__S__",str(S)).replace("__N__",str(N)).replace("__W__",str(W)).replace("__E__",str(E)))
IPython.display.HTML(_html)

In [ ]:
# Fine and Dead fuel moisture
# https://drive.google.com/file/d/1WgQbX0sRUxj4XR7hQffS1xYWflQ9Nc9u/view?usp=drive_link
show_drive_image("1WgQbX0sRUxj4XR7hQffS1xYWflQ9Nc9u", "/content/figures/dead_and_live_fuel_moisture.png", width=700)


In [ ]:
# Scott & Burgan scenarios
# https://drive.google.com/file/d/1BkOwaDEgYAgCpCWHw4NRXXP9CnhCyWVS/view?usp=drive_link
show_drive_image("1BkOwaDEgYAgCpCWHw4NRXXP9CnhCyWVS", "/content/figures/scott_burgan_scenarios.png", width=700)


---

# Activity 2 — Belgium wildfire case studies (30 min)

Simulate four **real 2025 Belgian wildfires** and compare each simulated fire scar
against the **observed perimeter**, sweeping fuel-moisture scenarios to find the
best fit. Goal: show Cell2Fire-W can reproduce observed fire spread in Belgium.

Each instance is centred on the real ignition; the observed scar lives in
`ground_truth/observed_scar.shp`.


In [ ]:
# https://drive.google.com/file/d/1bYQpUfWdB7AEdmd85oIm0cMtYCzm8Lp2/view?usp=sharing
show_drive_image("1bYQpUfWdB7AEdmd85oIm0cMtYCzm8Lp2", "/content/figures/weather_sampling.png", width=800)


In [ ]:
# @title
# helpers
import importlib, subprocess, sys
for pkg,mod in [("pyshp","shapefile"),("folium","folium"),("pyproj","pyproj"),("matplotlib","matplotlib")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",pkg])
# import numpy as np, matplotlib.pyplot as plt, csv, shutil, glob, subprocess
from matplotlib.colors import ListedColormap
from matplotlib.path import Path as MplPath
import shapefile
print("deps ready")

In [ ]:
BE = Path("c2f_be/C2F-W/data/handson/Instances_Belgium")
#CASE = "Case1_Clinge"        # Case1_Clinge | Case2_Houffalize | Case3_Arlon | Case4_Maasmechelen
#INST = BE/CASE
#WORK = Path("/tmp/c2f_be_runs")/CASE

#fuels,h = read_asc(INST/"fuels.asc")          # read_asc() ya definido en la Sec. 0 (se reutiliza aqui)
#NCOLS,NROWS = int(h["ncols"]),int(h["nrows"])
#CENTER = (NROWS//2)*NCOLS + NCOLS//2 + 1     # centre cell id (1-based)
#CRS = "EPSG:3812"                             # Belge Lambert 2008 (all Belgium cases)
#print(f"{CASE}: {NCOLS}x{NROWS} | cell {h['cellsize']:.0f} m | centre cell {CENTER} | CRS {CRS}")


### 2.1 Visual Comparson

In [ ]:
import io, base64, math
from PIL import Image
from matplotlib.patches import Ellipse as _Ellipse
import folium, pyproj

# best-fit moisture per case (from the §5 sweep); tweak freely
BEST_BY_CASE = {"Case1_Clinge":"D4L2","Case2_Houffalize":"D1L3",
                "Case3_Arlon":"D2L2","Case4_Maasmechelen":"D4L3"}

def _arrival_grid(msg_csv, center, nc, nr):
    arcs=[(int(r[0]),int(r[1]),float(r[2])) for r in csv.reader(open(msg_csv)) if len(r)>=4]
    ign={center:0.0}
    for i,j,tj in arcs: ign[j]=min(ign.get(j,1e18),tj)
    T=np.full((nr,nc),np.nan)
    for c,t in ign.items(): T[(c-1)//nc,(c-1)%nc]=t
    return T

def _huygens_gif(out, center, hc, crop, frames_n=46, hold=8, W=6.6, wlen=1.0, lw=1.7):
    # Forward Huygens wavelets at the advancing front: one ellipse per leading-edge cell,
    # rear focus at the cell, oriented downwind (source->cell), eccentricity from ROS.
    NCOLS,NROWS=int(hc["ncols"]),int(hc["nrows"]); r0,r1,c0,c1=crop
    arcs=[(int(r[0]),int(r[1]),float(r[2]),float(r[3]))
          for r in csv.reader(open(out/"Messages"/"MessagesFile1.csv")) if len(r)>=4]
    Tj={center:0.0}; src={}
    for i,j,tj,ros in arcs:
        if tj<Tj.get(j,1e18): Tj[j]=tj; src[j]=(i,ros)
    rv=[a[3] for a in arcs]; rlo,rhi=min(rv),max(rv)
    Tig=np.full((NROWS,NCOLS),np.nan)
    for c,ti in Tj.items(): Tig[(c-1)//NCOLS,(c-1)%NCOLS]=ti
    def xy(c): rr=(c-1)//NCOLS; cc=(c-1)%NCOLS; return cc+0.5, NROWS-rr-0.5
    def ecc(ros): return 0.45+0.45*(ros-rlo)/(rhi-rlo+1e-9)
    DARK=np.array([0x33,0x22,0x1b])/255
    tmax=np.nanmax(Tig); T=tmax*1.04; step=T/frames_n; band=step*5.0
    times=list(np.linspace(0,T,frames_n))+[T]*hold
    fig=plt.figure(figsize=(W,W*(r1-r0)/(c1-c0)),dpi=120); ax=fig.add_axes([0,0,1,1]); fig.patch.set_alpha(0)
    def burned(t):
        b=(Tig<=t); img=np.zeros((NROWS,NCOLS,4)); img[b,:3]=DARK; img[b,3]=0.80; return img
    def draw(t):
        ax.cla(); ax.set_xlim(c0,c1); ax.set_ylim(NROWS-r1,NROWS-r0); ax.axis("off"); ax.patch.set_alpha(0)
        ax.imshow(burned(t),extent=[0,NCOLS,0,NROWS],origin="upper",interpolation="nearest",zorder=2)
        for c in [c for c,ti in Tj.items() if (t-band)<=ti<=t and c in src]:
            i,ros=src[c]; xc,yc=xy(c); xi,yi=xy(i)
            ang=math.atan2(yc-yi,xc-xi) if (xc!=xi or yc!=yi) else 0.0
            e=ecc(ros); L=wlen*(0.6+0.8*(ros-rlo)/(rhi-rlo+1e-9)); bb=L*math.sqrt(1-e*e); foc=L*e
            glow=min(1.0,(t-Tj[c])/max(band,1e-6))
            ax.add_patch(_Ellipse((xc+foc*math.cos(ang),yc+foc*math.sin(ang)),2*L,2*bb,
                angle=math.degrees(ang),facecolor=(1.0,0.55,0.12,0.18),
                edgecolor=(1.0,0.78-0.35*glow,0.18,0.95),lw=lw,zorder=4,antialiased=True))
    rgba=[]
    for t in times:
        draw(t); fig.canvas.draw(); rgba.append(np.asarray(fig.canvas.buffer_rgba()).copy())
    plt.close(fig)
    #pil=[Image.fromarray(f,"RGBA") for f in rgba]
    pil=[Image.fromarray(f) for f in rgba]
    master=pil[len(pil)//2].convert("RGB").quantize(colors=255,method=Image.MEDIANCUT)
    frames=[]
    for im in pil:
        a=im.getchannel("A"); q=im.convert("RGB").quantize(palette=master,dither=Image.NONE)
        q.paste(255,a.point(lambda v:255 if v<55 else 0)); q.info["transparency"]=255; frames.append(q)
    buf=io.BytesIO()
    frames[0].save(buf,format="GIF",save_all=True,append_images=frames[1:],
                   duration=120,loop=0,transparency=255,disposal=2,optimize=False)
    return buf.getvalue()

def _bounds_crop(hd, crop, crs="EPSG:3812"):
    tw=pyproj.Transformer.from_crs(crs,"EPSG:4326",always_xy=True)
    xll,yll,cs=hd["xllcorner"],hd["yllcorner"],hd["cellsize"]; nr=int(hd["nrows"])
    r0,r1,c0,c1=crop; x0=xll+c0*cs; x1=xll+c1*cs; y1=yll+(nr-r0)*cs; y0=yll+(nr-r1)*cs
    la,lo=[],[]
    for x in (x0,x1):
        for y in (y0,y1):
            o,a=tw.transform(x,y); lo.append(o); la.append(a)
    return [[min(la),min(lo)],[max(la),max(lo)]]

def _perimeter_latlon(shp_path, crs="EPSG:3812"):
    tw=pyproj.Transformer.from_crs(crs,"EPSG:4326",always_xy=True)
    try: r=shapefile.Reader(str(shp_path),encodingErrors="replace")
    except Exception:
        sp=str(shp_path); r=shapefile.Reader(shp=open(sp,"rb"),shx=open(sp.replace(".shp",".shx"),"rb"))
    rings=[]
    for sh in r.shapes():
        P=sh.points; parts=list(sh.parts)+[len(P)]
        for k in range(len(parts)-1):
            rings.append([[a,o] for (o,a) in (tw.transform(x,y) for x,y in P[parts[k]:parts[k+1]])])
    return rings

def project_fire(case, scenario=None, margin=12, hours=None):
    inst=BE/case; _,hc=read_asc(inst/"fuels.asc")
    nc,nr=int(hc["ncols"]),int(hc["nrows"]); center=(nr//2)*nc+nc//2+1
    sc=scenario or BEST_BY_CASE.get(case,"D2L2")
    inst.joinpath("Ignitions.csv").write_text(f"Year,Ncell\n1,{center}\n")
    # (1) ventana = largo real del Weather.csv (FPL=1 -> 1 fila = 60 períodos)
    wrows=sum(1 for _ in open(inst/"Weather.csv"))-1
    nper=int(hours*60) if hours else wrows*60
    out=Path("/tmp/proj")/case
    if out.exists(): shutil.rmtree(out)
    out.mkdir(parents=True)
    cmd=[EXEC,"--input-instance-folder",str(inst)+"/","--output-folder",str(out)+"/",
        "--sim","S","--ignitions","--seed","1","--nsims","1","--final-grid","--output-messages",
        "--moisture-scenario",sc,"--Fire-Period-Length","1.0","--Weather-Period-Length","60",
        "--max-fire-periods",str(nper)]                       # <- ya no 160 fijo
    # (2) reintento: la 1era corrida "en frío" a veces no escribe Messages
    for _ in range(3):
        subprocess.run(cmd,capture_output=True,text=True)
        if (out/"Messages/MessagesFile1.csv").exists(): break
    T=_arrival_grid(out/"Messages/MessagesFile1.csv",center,nc,nr)
    rs,csx=np.where(np.isfinite(T))
    r0,r1=max(rs.min()-margin,0),min(rs.max()+margin,nr)
    c0,c1=max(csx.min()-margin,0),min(csx.max()+margin,nc)
    gif=_huygens_gif(out,center,hc,(r0,r1,c0,c1)); uri="data:image/gif;base64,"+base64.b64encode(gif).decode()
    b=_bounds_crop(hc,(r0,r1,c0,c1)); ctr=[(b[0][0]+b[1][0])/2,(b[0][1]+b[1][1])/2]
    fmap=folium.Map(location=ctr,zoom_start=16,tiles=None,control_scale=True)
    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                     name="satellite",attr="Esri").add_to(fmap)
    folium.TileLayer("OpenStreetMap",name="OSM").add_to(fmap)
    for ring in _perimeter_latlon(inst/"ground_truth/observed_scar.shp"):
        folium.PolyLine(ring,color="#ff2d2d",weight=3,opacity=0.95,
                        tooltip="observed perimeter").add_to(fmap)
    folium.raster_layers.ImageOverlay(uri,bounds=b,opacity=0.92,
                                      name=f"Cell2Fire spread ({sc})",zindex=5).add_to(fmap)
    o,a=pyproj.Transformer.from_crs("EPSG:3812","EPSG:4326",always_xy=True).transform(
        hc["xllcorner"]+(center-1)%nc*hc["cellsize"]+hc["cellsize"]/2,
        hc["yllcorner"]+(nr-(center-1)//nc-0.5)*hc["cellsize"])
    folium.Marker([a,o],tooltip=f"ignition (cell {center})",
                  icon=folium.Icon(color="orange",icon="fire",prefix="fa")).add_to(fmap)
    folium.LayerControl(collapsed=False).add_to(fmap); fmap.fit_bounds(b)
    print(f"{case}: scenario {sc} | window={nper} min | tmax={np.nanmax(T):.0f} min | burned cells={int(np.isfinite(T).sum())}")
    return fmap

In [ ]:
print(BE)

In [ ]:
inst = "Case1_Clinge"
project_fire(inst,scenario="D4L2")

In [ ]:
project_fire("Case2_Houffalize",scenario = "D1L3")

In [ ]:
project_fire("Case3_Arlon",scenario = "D4L2")

### 2.2 Numerical comparison -- Intersection-over-Union (Jaccard index) between simulated and observed scar

\begin{equation}
\mathrm{IoU}(S,O)
\;=\;
\frac{\lvert S \cap O \rvert}{\lvert S \cup O \rvert}
\;=\;
\frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FP}+\mathrm{FN}} ,
\end{equation}



In [ ]:
# https://drive.google.com/file/d/1oljnSI9hZ9V1oZsVDB1BB8kYjizPgjAO/view?usp=sharing
show_drive_image("1oljnSI9hZ9V1oZsVDB1BB8kYjizPgjAO", "/content/figures/fire_comparison.png", width=600)


In [ ]:
# https://drive.google.com/file/d/1vENkKALOh2GyDYZsXrIZRPWFevtALW9s/view?usp=drive_link
show_drive_image("1vENkKALOh2GyDYZsXrIZRPWFevtALW9s", "/content/figures/raterizing_fire_scar.png", width=500)

In [ ]:
def rasterize_perimeter(shp_path, header):
    sp = str(shp_path)
    try:    r = shapefile.Reader(shp=open(sp, "rb"), shx=open(sp.replace(".shp", ".shx"), "rb"))
    except Exception:
        sp=str(shp_path); r=shapefile.Reader(shp=open(sp,"rb"), shx=open(sp.replace(".shp",".shx"),"rb"))
    nc,nr=int(header["ncols"]),int(header["nrows"])
    xll,yll,cs=header["xllcorner"],header["yllcorner"],header["cellsize"]
    xs=xll+(np.arange(nc)+0.5)*cs; ys=yll+(nr-np.arange(nr)-0.5)*cs
    XX,YY=np.meshgrid(xs,ys); pts=np.column_stack([XX.ravel(),YY.ravel()])
    inside=np.zeros(pts.shape[0],bool)
    for sh in r.shapes():
        P=sh.points; parts=list(sh.parts)+[len(P)]; verts=[]; codes=[]
        for k in range(len(parts)-1):
            ring=P[parts[k]:parts[k+1]]
            if len(ring)<3: continue
            verts+=ring; codes+=[MplPath.MOVETO]+[MplPath.LINETO]*(len(ring)-2)+[MplPath.CLOSEPOLY]
        if verts: inside |= MplPath(verts,codes).contains_points(pts)
    return inside.reshape(nr,nc)


In [ ]:
CASE = "Case3_Arlon" #"Case1_Clinge" #"Case2_Houffalize" # "Case3_Arlon"
INST = BE/CASE
fuels,h = read_asc(INST/"fuels.asc")

OBS = rasterize_perimeter(INST/"ground_truth/observed_scar.shp", h)
print(f"observed burned cells: {int(OBS.sum())}  (~{OBS.sum()*h['cellsize']**2/1e4:.2f} ha)")

In [ ]:
import numpy as np

def iou_scar(sim, obs):
    """
    Intersection-over-Union (Jaccard) entre cicatriz simulada y observada.
    sim, obs : arrays booleanos de la MISMA grilla (True = celda quemada).
    Devuelve IoU y las métricas auxiliares (Dice, POD, FAR, áreas).
    """
    sim = np.asarray(sim, dtype=bool)
    obs = np.asarray(obs, dtype=bool)
    if sim.shape != obs.shape:
        raise ValueError("sim y obs deben tener la misma forma (misma grilla)")

    TP = int(np.sum( sim &  obs))     # quemada en ambos
    FP = int(np.sum( sim & ~obs))     # simulada quema, observada no  (sobre-predicción)
    FN = int(np.sum(~sim &  obs))     # observada quema, simulada no  (sub-predicción)

    iou  = TP / (TP + FP + FN) if (TP + FP + FN) else 0.0
    dice = 2*TP / (2*TP + FP + FN) if (2*TP + FP + FN) else 0.0
    pod  = TP / (TP + FN) if (TP + FN) else 0.0     # probability of detection (hit rate)
    far  = FP / (TP + FP) if (TP + FP) else 0.0     # false alarm ratio

    return dict(IoU=iou, Dice=dice, POD=pod, FAR=far, TP=TP, FP=FP, FN=FN)

In [ ]:
obs = OBS

sim = np.loadtxt(f"/tmp/proj/{CASE}/Grids/Grids1/ForestGrid0.csv", delimiter=",")
iou_scar(sim, obs)



---

# Activity 3 — Generating wildfire risk maps (30 min)

During each simulation, Cell2Fire records the burned cells and the fire-spread sequence. After running \(N\) simulations, the burn probability of each cell \(i\) is estimated as the fraction of simulations in which that cell burns:

$$
BP(i) = \frac{1}{N} \sum_{n=1}^{N} \mathbb{1}_{\{i \in B_n\}},
$$

where \(B_n\) is the set of burned cells in simulation \(n\), and \(\mathbb{1}_{\{i \in B_n\}}\) is equal to 1 if cell \(i\) burned in that simulation, and 0 otherwise.

The resulting Burn Probability Map (BPM) provides a spatial representation of the relative likelihood of fire occurrence across the landscape (the fraction of simulations in which a cell burned).

In [ ]:
# https://drive.google.com/file/d/1eCvQN1avN_KZjC1QFyMKo0q2QIizL0_S/view?usp=drive_link
show_drive_image("1eCvQN1avN_KZjC1QFyMKo0q2QIizL0_S", "/content/figures/bpmap_framework.png", width=500)


### 3.1 Gathering and sampling local historical weather conditions

In [ ]:
show_drive_image("1c3CyBfyUUdUbOStuB-SOQLiSLq3d-tnR", "/content/weather_sampling.png", width=800)


In [ ]:
# === weather_sampling: arma la carpeta Weathers/ para el ensemble de Burn Probability ===
# Muestrea VENTANAS consecutivas de nrows horas de tarde, de la climatología de verano ERA5.
import requests, pandas as pd, numpy as np
from pathlib import Path

ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"   # ERA5 (gratis, sin key, CC-BY)

def weather_sampling(inst_dir, lat, lon, n_weathers=30, nrows=5,
                     years=range(2015, 2025), months=(6,7,8,9),
                     afternoon=(12,19), tz="Europe/Madrid",
                     inst_name="JaimeCarrasco", seed=1):
    """
    Escribe inst_dir/Weathers/Weather1..N.csv, cada uno con `nrows` filas horarias
    = una ventana consecutiva de tarde de verano tomada de ERA5.
      n_weathers : nº de archivos (= réplicas del ensemble)
      nrows      : filas por archivo = HORAS DE SIMULACIÓN de cada réplica  (prueba 4–6)
      afternoon  : (h_ini, h_fin) hora local de INICIO de la ventana
    """
    rng = np.random.default_rng(seed)
    frames=[]
    for y in years:
        p = dict(latitude=lat, longitude=lon,
                 start_date=f"{y}-{min(months):02d}-01",
                 end_date=f"{y}-{max(months):02d}-30",
                 hourly="wind_speed_10m,wind_direction_10m,temperature_2m,relative_humidity_2m",
                 timezone=tz, wind_speed_unit="kmh")
        h = requests.get(ARCHIVE, params=p, timeout=60).json()["hourly"]
        frames.append(pd.DataFrame(h))
    df = pd.concat(frames, ignore_index=True)
    df["time"] = pd.to_datetime(df["time"])
    df = df.dropna(subset=["wind_speed_10m","wind_direction_10m"]).reset_index(drop=True)
    df["hour"]=df["time"].dt.hour; df["month"]=df["time"].dt.month
    df = df[df["month"].isin(months)]

    h0,h1 = afternoon; g = df.set_index("time"); starts=[]
    for t in df["time"]:
        if not (h0 <= t.hour <= h1): continue
        win = g.loc[t : t + pd.Timedelta(hours=nrows-1)]
        if len(win)==nrows and (win.index[-1]-win.index[0])==pd.Timedelta(hours=nrows-1):
            starts.append(t)
    if not starts:
        raise RuntimeError("no se encontraron ventanas válidas; revisa afternoon/nrows")

    wdir = Path(inst_dir)/"Weathers"; wdir.mkdir(parents=True, exist_ok=True)
    for f in wdir.glob("Weather*.csv"): f.unlink()
    picks = rng.choice(len(starts), size=min(n_weathers,len(starts)),
                       replace=len(starts)<n_weathers)
    for k,idx in enumerate(picks, start=1):
        t0=starts[idx]; win=g.loc[t0 : t0+pd.Timedelta(hours=nrows-1)]
        L=["Instance,datetime,WS,WD,T,RH"]
        for i,(ts,r) in enumerate(win.iterrows()):
            L.append(f"{inst_name},{ts:%Y-%m-%d %H:%M},{r.wind_speed_10m:.1f},"
                     f"{r.wind_direction_10m:.0f},{r.temperature_2m:.1f},{r.relative_humidity_2m:.0f}")
        (wdir/f"Weather{k}.csv").write_text("\n".join(L)+"\n")
    print(f"escritos {len(picks)} weathers de {nrows} filas en {wdir}")
    return wdir


In [ ]:
inst = "c2f_be/C2F-W/data/handson/vilo100"
weather_sampling(inst, lat=42.0976, lon=2.9806, n_weathers=100, nrows=5)

In [ ]:
!ls -1 /content/c2f_be/C2F-W/data/handson/vilo100/Weathers

### 3.2 Generating a BP Map with a constant moisture scenario

In [ ]:
# helper function
def _bp_from_grids(out, fuel, S, n):
    for g in glob.glob(str(out/"Grids"/"Grids*"/"ForestGrid*.csv")):
        G = np.loadtxt(g, delimiter=",")
        if G.shape == fuel.shape: S += (G > 0); n += 1
    return S, n

def _show_bp(BP, inst, title):
    fuel = np.loadtxt(Path(inst)/"fuels.asc", skiprows=6); nb = fuel <= 0
    plt.figure(figsize=(7.5, 6.6))
    plt.imshow(np.ma.masked_where(~nb, fuel), cmap="Greys", alpha=0.8)
    im = plt.imshow(np.ma.masked_where((BP <= 0) | nb, BP), cmap="YlOrRd", vmin=0, vmax=0.7)
    plt.colorbar(im, label="Burn Probability", fraction=0.046, pad=0.04)
    plt.title(title); plt.xticks([]); plt.yticks([]); plt.tight_layout(); plt.show()

def draw_ignitions(inst, nsims, seed):
    """nsims celdas de ignición aleatorias (quemables). Mismo seed -> mismas celdas en fixed y variable."""
    fuel = np.loadtxt(Path(inst)/"fuels.asc", skiprows=6); nc = fuel.shape[1]
    burn = np.argwhere(fuel > 0); rng = np.random.default_rng(seed)
    return [int(r)*nc + int(c) + 1 for r, c in burn[rng.integers(len(burn), size=nsims)]]


In [ ]:
# 1: Main function
import numpy as np, glob, subprocess, shutil, tempfile
from pathlib import Path
import matplotlib.pyplot as plt

def burn_probability_fixed(inst, nsims=100, scenario="D2L2", nrows=5, seed=1):
    """BP con el MISMO Dk para todos los incendios. Igniciones fijadas por seed; weather aleatorio."""
    inst = Path(inst); fuel = np.loadtxt(inst/"fuels.asc", skiprows=6)
    nw = len(list((inst/"Weathers").glob("Weather*.csv")))
    cells = draw_ignitions(inst, nsims, seed)
    out = Path(tempfile.gettempdir())/"bp_fixed"; S = np.zeros_like(fuel, float); n = 0
    for cell in cells:
        (inst/"Ignitions.csv").write_text(f"Year,Ncell\n1,{cell}\n")           # ignicion de esta replica
        if out.exists(): shutil.rmtree(out)
        out.mkdir(parents=True)
        cmd = [EXEC, "--input-instance-folder", str(inst)+"/", "--output-folder", str(out)+"/", "--sim", "S",
               "--ignitions", "--nsims", "1", "--weather", "random", "--nweathers", str(nw),
               "--moisture-scenario", scenario, "--final-grid", "--seed", str(seed),
               "--Fire-Period-Length", "1.0", "--Weather-Period-Length", "60", "--max-fire-periods", str(nrows*60)]
        for _ in range(3):
            subprocess.run(cmd, capture_output=True, text=True)
            S, n_new = _bp_from_grids(out, fuel, S, n)      # helper ya definido arriba (antes no se usaba)
            if n_new > n: n = n_new; break
    BP = S/max(n, 1)
    _show_bp(BP, inst, f"Burn Probability - FIXED {scenario}  ({n} sims)")
    return BP


In [ ]:
# 2: Showing the BP
inst = "c2f_be/C2F-W/data/handson/vilo100"
BP50 = burn_probability_fixed(inst, nsims=50)

In [ ]:
# 3: BP-Map projection on an interactive map

import numpy as np, folium, pyproj
import matplotlib.pyplot as plt
from pathlib import Path
# options for cmap: "Reds", "hot_r", "autumn_r", or "YlOrRd"
def bp_to_folium(BP, inst, crs = "EPSG:25831", opacity = 0.6, cmap = "YlOrRd"):
    _, h = read_asc(Path(inst)/"fuels.asc")           # reutiliza el parser de la Sec. 0
    nc,nr,cs=int(h["ncols"]),int(h["nrows"]),h["cellsize"]; xll,yll=h["xllcorner"],h["yllcorner"]
    # esquinas del dominio -> lat/lon (bbox)
    tw=pyproj.Transformer.from_crs(crs,"EPSG:4326",always_xy=True)
    corners=[(xll,yll),(xll+nc*cs,yll),(xll,yll+nr*cs),(xll+nc*cs,yll+nr*cs)]
    lls=[tw.transform(x,y) for x,y in corners]
    lats=[p[1] for p in lls]; lons=[p[0] for p in lls]
    south,north,west,east=min(lats),max(lats),min(lons),max(lons)
    # overlay RGBA: color por BP, transparente donde no quema o no-combustible
    fuel=np.loadtxt(Path(inst)/"fuels.asc",skiprows=6); nb=fuel<=0
    vmax=BP.max() if BP.max()>0 else 1
    rgba=plt.get_cmap(cmap)(np.clip(BP/vmax,0,1)); rgba[...,3]=np.where((BP>0)&(~nb),1.0,0.0)
    img=(rgba*255).astype("uint8")
    # mapa
    m=folium.Map(location=[(south+north)/2,(west+east)/2], zoom_start=14, tiles=None)
    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                     attr="Esri", name="Satélite").add_to(m)
    folium.TileLayer("OpenStreetMap",name="OSM").add_to(m)
    folium.raster_layers.ImageOverlay(image=img, bounds=[[south,west],[north,east]],
                     opacity=opacity, name=f"Burn Probability (máx {vmax:.2f})").add_to(m)
    folium.LayerControl().add_to(m)
    m.fit_bounds([[south,west],[north,east]])
    return m

m = bp_to_folium(BP50, inst)   # BP_fix = array del ensemble; vilo = ruta a vilo100
m

3.3 Generating a BP Map with a variable moisture scenario

Now, we assign an S&B moisture scenario (DkL2) to each fire simulation based on T (°C) and RH (%), using Simard’s equation:
$$
T_f = \frac{9}{5}T + 32
$$
$$
\mathrm{EMC}(T,\mathrm{RH}) =
\begin{cases}
0.03229 + 0.281073\,\mathrm{RH} - 0.000578\,\mathrm{RH}\,T_f, & \mathrm{RH} < 10, \\
2.22749 + 0.160107\,\mathrm{RH} - 0.014784\,T_f, & 10 \leq \mathrm{RH} < 50, \\
21.0606 + 0.005565\,\mathrm{RH}^{2} - 0.00035\,\mathrm{RH}\,T_f - 0.483199\,\mathrm{RH}, & \mathrm{RH} \geq 50.
\end{cases}
$$
where $T_f$ is the air temperature expressed in degrees Fahrenheit, obtained from the air temperature $T$ in degrees Celsius, and RH the air relative humidity.

In [ ]:
FILE_ID = "1x-8wKRPotSQygnsiACgZGqG4S6hPmpjt"

!wget -q -O /content/weather_sampling.png "https://drive.google.com/uc?export=download&id={FILE_ID}"

from IPython.display import Image, display
display(Image("/content/weather_sampling.png", width=500))

In [ ]:
import numpy as np, glob, csv, subprocess, shutil, tempfile
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt

# --- EMC (Simard 1968) -> nivel de humedad muerta Dk (D1=seco ... D4=humedo) ---
def emc(T, RH):
    Tf = T*9/5 + 32
    if RH < 10: return 0.03229 + 0.281073*RH - 0.000578*RH*Tf
    if RH < 50: return 2.22749 + 0.160107*RH - 0.014784*Tf
    return 21.0606 + 0.005565*RH**2 - 0.00035*RH*Tf - 0.483199*RH

def emc_to_dk(e): return 1 if e < 4.5 else 2 if e < 7.5 else 3 if e < 10.5 else 4

def weather_dk(wf):
    rows = list(csv.DictReader(open(wf)))
    return emc_to_dk(np.mean([emc(float(r["T"]), float(r["RH"])) for r in rows]))

def burn_probability_variable(inst, nsims=100, live="L2", nrows=5, seed=1):
    """BP con Dk segun el EMC (T/RH) del weather de cada incendio. MISMAS igniciones que fixed (mismo seed)."""
    inst = Path(inst); fuel = np.loadtxt(inst/"fuels.asc", skiprows=6)
    wfiles = sorted((inst/"Weathers").glob("Weather*.csv"))
    cells = draw_ignitions(inst, nsims, seed)
    rng = np.random.default_rng(seed)                                          # elige un weather por incendio
    out = Path(tempfile.gettempdir())/"bp_var"; S = np.zeros_like(fuel, float); n = 0; dks = []
    for cell in cells:
        wf = str(wfiles[rng.integers(len(wfiles))]); k = weather_dk(wf); dks.append(k)   # su Dk desde EMC
        (inst/"Ignitions.csv").write_text(f"Year,Ncell\n1,{cell}\n")
        shutil.copy(wf, inst/"Weather.csv")
        if out.exists(): shutil.rmtree(out)
        out.mkdir(parents=True)
        cmd = [EXEC, "--input-instance-folder", str(inst)+"/", "--output-folder", str(out)+"/", "--sim", "S",
               "--ignitions", "--nsims", "1", "--moisture-scenario", f"D{k}{live}", "--final-grid",
               "--Fire-Period-Length", "1.0", "--Weather-Period-Length", "60", "--max-fire-periods", str(nrows*60)]
        for _ in range(3):
            subprocess.run(cmd, capture_output=True, text=True)
            S, n_new = _bp_from_grids(out, fuel, S, n)      # mismo helper que burn_probability_fixed
            if n_new > n: n = n_new; break
    BP = S/max(n, 1)
    print("Dk asignados (EMC):", {f"D{k}": Counter(dks).get(k, 0) for k in [1, 2, 3, 4]})
    _show_bp(BP, inst, f"Burn Probability - VARIABLE (Dk from EMC)  ({n} sims)")
    return BP


In [ ]:
inst = "c2f_be/C2F-W/data/handson/vilo100"
BP50v = burn_probability_variable(inst, nsims=50)


<div align="center">

# Thanks.

</div>

---
---


# GENERAL GOAL OF OUR RESEARCH

In [ ]:
# Goal:
#  https://drive.google.com/file/d/1aFjYqkoyKjo0TvpU3cvzsFEgbW-CvmaB/view?usp=drive_link

show_drive_image("1aFjYqkoyKjo0TvpU3cvzsFEgbW-CvmaB", "/content/figures/goal.png", width=900)

In [ ]:
# INTEGRATED, USABLE AND TRANSFERABLE SYSTEM

# https://drive.google.com/file/d/1W-46NvZv9bUdaAlUmt4ZWjP_nmOtaXh5/view?usp=drive_link
show_drive_image("1W-46NvZv9bUdaAlUmt4ZWjP_nmOtaXh5", "/content/figures/system.png", width=700)

## Some references

- Carrasco-Barra, J., González-Olabarria, J. R., Matus-Olivares, C., Ulloa-Fierro, F., Soto, F., Palacios, D., ... & Weintraub, A. (2026). New extensions of Cell2Fire software for fire risk analysis and evaluation in Chilean forests. Environmental Modelling & Software, 107009.
- Carrasco-Barra, J., Gonzalez-Olabarria, J. R., Palacios, D., Mahaluf, R., Garcia-Gonzalo, J., & Weintraub, A. (2025). Multicriteria firebreak planning for protecting ecological and cultural values under Wildfire risk: A case study in Catalonia. Environmental and Sustainability Indicators, 100956.
- Murray, L., Castillo, T., de Diego, I. M., Weber, R., Gonzalez-Olabarria, J. R., Garcia-Gonzalo, J., ... & Carrasco-Barra, J. (2025). Deep reinforcement learning for optimal firebreak placement in forest fire prevention. Applied Soft Computing, 175, 113043.
-Mancilla-Wulff, I., Terán, D., Vairetti, C., González-Olabarria, J. R., Weintraub, A., & Carrasco-Barra, J. (2025). A scalable AI-driven approach for burned-area mapping using U-Net and Landsat imagery. Applied Soft Computing, 114070.
- Carrasco, J., Mahaluf, R., Lisón, F., Pais, C., Miranda, A., de la Barra, F., ... & Weintraub, A. (2023). A firebreak placement model for optimizing biodiversity protection at landscape scale. Journal of Environmental Management, 342, 118087.